In [ ]:
#Brihanna Enright (23362901), Ilja Blosenko (23375493), HuiLin Chong (23365684)
#State if the code runs end to end without error
#Any third-party links

# Group 22 Assignment 2: Implementing a Reinforcement Learning agent using a Deep Q Network
## 

In [1]:
#Imports
!pip install gymnasium
import gymnasium as gym     #imports the gym environment from OpenAI's library


   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ---------------


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\brien\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Section 1: Why is Reinforcement Learning used for this task?

## Section 2: Our Gym Environment
We have chosen to implement the MountainCar-v0 gym environment from the Gymnasium Library [2]. The Mountain Car is a deterministic MDP that randomly places a car at the bottom of a sinusoidal valley. The goal is to strategically accelerate the car to reach the goal state on top of the right hill. 

The agent can make two observations of its environment at any given time. The observation space is a ndarray with shape (2,)
* Position of the car along the x axis. Min: -1.2. Max: 0.6
* Velocity of the car. Min: -0.07. Max: 0.07

There are three discrete deterministic actions the agent can take:
* 0: Accelerate to the left
* 1: Don't accelerate
* 2: Accelerate to the right

Given an action, the mountain car follows the following transitions
* velocityt+1 = velocityt + (action - 1) * force - cos(3 * positiont) * gravity
* positiont+1 = positiont + velocityt+1

where force = 0.001 and gravity = 0.0025. The velocity is set to 0 on collision with the wall. 

The goal is to reach the flag placed on top of the right hill as quickly as possible. The agent is penalised with a reward of -1 for each timestep. The episode ends if the position of the car is greater than or equal to 0.5 or the length of the episode is 200.

In [ ]:
environment = gym.make("MountainCar-v0")

## Section 3: Implementation

### 3.1 Capture and Sampling of Data

### 3.2 The Network Structure

MountainCar gives the agent two measurements at each step, its position and velocity. The same position can call for different actions depending where the car is moving. A fully connected network lets us combine these measurements to estimate the value of accelerating left, coasting, or accelerating right.

We use two hidden layers of 64 units as a starting design for this small numerical input. Each hidden layer uses ReLU so the network can learn relationships that change across the valley. The three output units each predict the discounted sum of future rewards for taking one action and then following the greedy policy. MountainCar charges one reward point per step, so a less negative return is better.

In [ ]:
import torch
from torch import nn

# Section 2 creates "environment". Read the sizes from it so the model
# accepts its observation vector and produces one value per available action.
n_observations = environment.observation_space.shape[0]  # position and velocity
n_actions = environment.action_space.n                  # left, coast, right

class QNetwork(nn.Module):
    def __init__(self, input_size, action_count):
        super().__init__()
        self.layers = nn.Sequential(
            # Each of the 64 units learns its own combination of position
            # and velocity. ReLU keeps positive activations and zeros the rest.
            nn.Linear(input_size, 64),
            nn.ReLU(),
            # Combine those features again so action values can vary with
            # both the car's location and the direction of its momentum.
            nn.Linear(64, 64),
            nn.ReLU(),
            # Output indices match the environment's action IDs (0, 1, 2).
            # A linear output allows negative Q-values, as expected from
            # the -1 step reward. These values do not need to sum to one.
            nn.Linear(64, action_count),
        )

    def forward(self, states):
        # A single [position, velocity] gives three Q-values. A batch of
        # observations gives a separate row of three Q-values for each car state.
        return self.layers(states)


# Create these once before training. Section 3.3 will update the online
# network's weights, while the target network supplies reference predictions.
online_net = QNetwork(n_observations, n_actions)
target_net = QNetwork(n_observations, n_actions)

# Copy the initial weights into a separate model. Assigning target_net =
# online_net would share the weights and both would change at every update.
target_net.load_state_dict(online_net.state_dict())
target_net.requires_grad_(False)  # Only the online model needs gradients.
target_net.eval()                 # Use the target model for inference.

print(online_net)

In [ ]:
# Check a valid example state without resetting the group's environment.
# The car is at position -0.5 and is stationary. One row means one observation.
example_state = torch.tensor([[-0.5, 0.0]], dtype=torch.float32)
with torch.no_grad():
    online_values = online_net(example_state)
    target_values = target_net(example_state)

# The model should produce one row with three columns. Immediately after
# copying the weights, the two networks should agree on this observation.
assert online_values.shape == (1, n_actions)
torch.testing.assert_close(online_values, target_values)

print("Input shape:", tuple(example_state.shape))
for action_name, value in zip(("left", "coast", "right"), online_values[0]):
    print(f"{action_name:>5}: {value.item(): .4f}")
print("Initial online and target predictions match.")

# These are predictions from random initial weights. Their ranking at this
# point says nothing about which action will work best after training.

### 3.3 Discussion of the Code
(where Q learning is applied to the weights)

## Section 4: Results

## Section 5: Evaluation of Results

## Section 6: Independent Research

## Section 7: Statement of Work

### Brihanna Enright (23362901)
* 20/11/2026 - Done the setup for the team
* 23/11/2026 - Implemented section 2 - the gym environment

### Ilja Blosenko (23375493)
* 24/11/2026 - Implemented section 3.2

### HuiLin Chong (23365684)

## Section 8: Use of Generative AI

### Brihanna Enright (23362901)
* Attached the assignment spec, along with the following prompt: Here is my assignment for neural computing. I think my group is going to choose the easier option 1. Can you explain what gym environments there are, what classic control is and problems other than the pole balancing one. **AI Response** Claude responded with an explanation of the Gymnasium library and gave a list of the problems to choose from. It also gave reasons as to why we should choose each one. We used this to research each problem and decide which we should choose.
* Prompted what is meant by capturing and sampling of data in relation to MountainCar-v0? **AI Response** Claude responded with an explanation of capture (saving each transition) and sampling (training on the experience)

### Ilja Blosenko (23375493)

### HuiLin Chong (23365684)

## Section 9: References

[1] Gymnasium Documentation, Farama Foundation. Last updated 05/10/2026. Available: https://gymnasium.farama.org/introduction/basic_usage/

[2] Gymnasium Documentation, Farama Foundation. Available: https://gymnasium.farama.org/environments/classic_control/mountain_car/